In [17]:
import pandas as pd
import numpy as np
import glob

import os, glob
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

files = glob.glob("*.csv")

df_list = []
for file in files:
    temp = pd.read_csv(file)
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

df.columns = df.columns.str.strip()

# remove duplicate columns
df = df.loc[:, ~df.columns.duplicated()]

# remove duplicate rows
df = df.drop_duplicates()

# replace inf
df = df.replace([np.inf, -np.inf], np.nan)

# drop empty columns
df = df.dropna(axis=1, how="all")

# remove constant columns
constant_cols = df.columns[df.nunique() <= 1]
df = df.drop(columns=constant_cols)

# recreate X and y AFTER cleaning df
X = df.drop("Label", axis=1)
X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median())

y = df["Label"].apply(lambda x: 0 if x == "BENIGN" else 1)

print("Final X shape:", X.shape)
print("Final y counts:")
print(y.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train:", X_train_scaled.shape)
print("X_test:", X_test_scaled.shape)
print("y_train:")
print(y_train.value_counts(normalize=True))
print("y_test:")
print(y_test.value_counts(normalize=True))

lr = LogisticRegression(max_iter=1000, n_jobs=-1)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

print("LR Accuracy:", accuracy_score(y_test, y_pred_lr))
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Final X shape: (2436679, 70)
Final y counts:
Label
0    2099662
1     337017
Name: count, dtype: int64
X_train: (1949343, 70)
X_test: (487336, 70)
y_train:
Label
0    0.86169
1    0.13831
Name: proportion, dtype: float64
y_test:
Label
0    0.861691
1    0.138309
Name: proportion, dtype: float64
LR Accuracy: 0.9748941182264392
[[419043    890]
 [ 11345  56058]]
              precision    recall  f1-score   support

           0       0.97      1.00      0.99    419933
           1       0.98      0.83      0.90     67403

    accuracy                           0.97    487336
   macro avg       0.98      0.91      0.94    487336
weighted avg       0.98      0.97      0.97    487336

RF Accuracy: 0.9994459674639263
[[419846     87]
 [   183  67220]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    419933
           1       1.00      1.00      1.00     67403

    accuracy                           1.00    487336
   macro avg       1.00  